# AuraGateway vLLM CUDA 12.9 resolution reconnaissance v1

Resolve and inventory the complete dependency closure without installing packages, writing wheel artifacts, loading a model, or issuing requests.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

NOTEBOOK_NAME = "auragateway-cu129-resolution-reconnaissance-v1"
OUTPUT_DIRECTORY_NAME = "auragateway_vllm_cu129_resolution_reconnaissance_v1"
OUTPUT_ROOT = Path("/kaggle/working") / OUTPUT_DIRECTORY_NAME

VLLM_RELEASE = "0.19.1"
VLLM_DISTRIBUTION = "0.19.1"
VLLM_RELEASE_API = (
    "https://api.github.com/repos/vllm-project/vllm/releases/tags/v0.19.1"
)
VLLM_ASSET_NAME = "vllm-0.19.1-cp38-abi3-manylinux_2_31_x86_64.whl"
VLLM_ASSET_SHA256 = (
    "71a87f46cafab4489c69a5c5c83b870d0235e5694d8222303d460576293dc719"
)
PYPI_INDEX = "https://pypi.org/simple"
PYTORCH_INDEX = "https://download.pytorch.org/whl/cu129"
FIXED_REQUIREMENTS = (
    "torch==2.10.0+cu129",
    "torchaudio==2.10.0+cu129",
    "torchvision==0.25.0+cu129",
    "transformers==5.5.3",
)
APPROVED_HOST_AUTHORITIES = {
    "download.pytorch.org": "pytorch",
    "download-r2.pytorch.org": "pytorch",
    "files.pythonhosted.org": "pypi",
    "github.com": "github_release",
    "objects.githubusercontent.com": "github_release",
    "release-assets.githubusercontent.com": "github_release",
}
CANDIDATE_HOST_AUTHORITIES = {
    "pypi.nvidia.com": "nvidia",
}
CREDENTIAL_ENV_NAMES = (
    "ANTHROPIC_API_KEY",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "GOOGLE_API_KEY",
    "HF_TOKEN",
    "HUGGING_FACE_HUB_TOKEN",
    "OPENAI_API_KEY",
    "OPENROUTER_API_KEY",
)
HISTORICAL_FAILURES = (
    {
        "attempt": 1,
        "classification": "MATERIALIZER_RELEASE_ASSET_CONTRACT_FAILURE",
        "code": "VLLM_CU128_RELEASE_ASSET_ABSENT",
        "execution_log_sha256": (
            "b45bee3fd286f35d367ee25639100eb33b9244251d5a921dedd84c998e785a2d"
        ),
        "historical_kaggle_title": "auragateway-cu128-wheelhouse-asset-mismatch-v1",
        "first_divergence": "official_v0.19.1_release_has_no_explicit_cu128_x86_64_wheel",
    },
    {
        "attempt": 2,
        "classification": "MATERIALIZER_DOWNLOAD_HOST_ALLOWLIST_FAILURE",
        "code": "PYTORCH_CDN_HOST_NOT_ALLOWED",
        "execution_log_sha256": (
            "69c7656374fc5313becb44684f1b11eac950db7c79eed5b62572eaefec3640a3"
        ),
        "historical_kaggle_title": "auragateway-cu129-wheelhouse-cdn-mismatch-v1",
        "first_divergence": "resolved_torch_wheel_uses_download-r2.pytorch.org",
    },
    {
        "attempt": 3,
        "classification": "MATERIALIZER_ACQUISITION_POLICY_FAILURE",
        "code": "NVIDIA_PACKAGE_HOST_NOT_ALLOWED",
        "execution_log_sha256": (
            "f6e6f844ebfb7ede0aab428e4766af4123622fb2f3092933e4070e26d6831fa4"
        ),
        "historical_kaggle_title": "auragateway-cu129-wheelhouse-nvidia-host-mismatch-v1",
        "first_divergence": "resolved_nvidia_cublas_wheel_uses_pypi.nvidia.com",
    },
)
HISTORICAL_RUNTIME_CONTEXT = {
    "execution_authority": "REJECTED",
    "diagnostic_admissibility": "HIGH",
    "observed_vllm": "0.25.1+cu129",
    "observed_vllm_wheel_sha256": (
        "9e206f370c934a2d4b6b1f05d3d09708d344e05d80260189ef19f60755709431"
    ),
    "observed_torch": "2.10.0+cu128",
    "required_torch_from_pip_check": "2.11.0",
    "observed_transformers": "5.0.0",
    "minimum_transformers_from_pip_check": "5.5.3",
    "native_import_error": "undefined symbol: torch_from_blob",
}
STATIC_MATERIALIZER_FINDINGS = (
    {
        "code": "MATERIALIZER_REQUIRED_PREFIX_VARIANT_DRIFT",
        "severity": "blocking",
        "bound_materializer_sha256": (
            "a3e043ba6c2caf982a0ebe14ddd1d102e0b5066a46ff17f6fdbf7e0bf876cf79"
        ),
        "evidence": (
            "the cu129 materializer still checks vllm, torch, torchaudio, and torchvision "
            "filename prefixes containing cu128"
        ),
        "required_resolution": (
            "repair all required wheel filename prefixes together with the complete "
            "reviewed source-authority policy after reconnaissance"
        ),
    },
)


def canonical_json(payload: object) -> str:
    return json.dumps(payload, ensure_ascii=True, separators=(",", ":"), sort_keys=True)


def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def normalize_distribution_name(value: str) -> str:
    return value.lower().replace("_", "-")[:120]


def archive_sha256(archive_info: dict[str, object]) -> str | None:
    hash_value = archive_info.get("hash")
    if isinstance(hash_value, str) and hash_value.startswith("sha256="):
        digest = hash_value.removeprefix("sha256=")
        if len(digest) == 64:
            return digest
    hashes = archive_info.get("hashes")
    if isinstance(hashes, dict):
        sha256_value = hashes.get("sha256")
        if isinstance(sha256_value, str) and len(sha256_value) == 64:
            return sha256_value
    return None


def expected_authority(distribution_name: str) -> str:
    normalized = normalize_distribution_name(distribution_name)
    if normalized == "vllm":
        return "github_release"
    if normalized in {"torch", "torchaudio", "torchvision"}:
        return "pytorch"
    if normalized.startswith("nvidia-"):
        return "nvidia"
    return "pypi"


def observed_authority(hostname: str) -> tuple[str, str]:
    if hostname in APPROVED_HOST_AUTHORITIES:
        return APPROVED_HOST_AUTHORITIES[hostname], "approved"
    if hostname in CANDIDATE_HOST_AUTHORITIES:
        return CANDIDATE_HOST_AUTHORITIES[hostname], "review_required"
    return "unknown", "review_required"


def sanitized_url_parts(value: object) -> dict[str, object]:
    if not isinstance(value, str):
        return {
            "scheme": "missing",
            "hostname": "missing",
            "artifact_filename": "missing",
            "path_prefix": "/",
            "sanitized_url": None,
            "url_sha256": None,
            "credentials_present": False,
            "fragment_present": False,
        }
    parsed = urllib.parse.urlsplit(value)
    hostname = parsed.hostname or "missing"
    path = PurePosixPath(parsed.path)
    path_parts = tuple(part for part in path.parts if part != "/")
    path_prefix = "/" + "/".join(path_parts[:2]) if path_parts else "/"
    sanitized_url = urllib.parse.urlunsplit(
        (parsed.scheme, hostname, parsed.path, "", "")
    )
    return {
        "scheme": parsed.scheme or "missing",
        "hostname": hostname,
        "artifact_filename": path.name or "missing",
        "path_prefix": path_prefix,
        "sanitized_url": sanitized_url,
        "url_sha256": sha256_text(value),
        "credentials_present": (
            parsed.username is not None or parsed.password is not None
        ),
        "fragment_present": bool(parsed.fragment),
    }


def evaluate_install_record(
    item: object,
    *,
    index: int,
) -> tuple[dict[str, object], list[dict[str, object]]]:
    violations: list[dict[str, object]] = []
    if not isinstance(item, dict):
        record = {
            "record_index": index,
            "distribution_name": f"invalid-record-{index}",
            "normalized_name": f"invalid-record-{index}",
            "version": "missing",
            "expected_authority": "unknown",
            "observed_authority": "unknown",
            "authority_state": "review_required",
            "scheme": "missing",
            "hostname": "missing",
            "artifact_filename": "missing",
            "path_prefix": "/",
            "sanitized_url": None,
            "url_sha256": None,
            "sha256": None,
        }
        violations.append(
            {
                "record_index": index,
                "distribution_name": record["distribution_name"],
                "hostname": "missing",
                "failure_code": "RESOLUTION_RECORD_INVALID",
                "reason": "pip install record is not one JSON object",
            }
        )
        return record, violations

    metadata = item.get("metadata")
    download_info = item.get("download_info")
    if not isinstance(metadata, dict):
        metadata = {}
        violations.append(
            {
                "record_index": index,
                "distribution_name": f"unknown-record-{index}",
                "hostname": "missing",
                "failure_code": "RESOLUTION_METADATA_MISSING",
                "reason": "pip install record lacks metadata",
            }
        )
    if not isinstance(download_info, dict):
        download_info = {}
        violations.append(
            {
                "record_index": index,
                "distribution_name": str(metadata.get("name", f"unknown-record-{index}")),
                "hostname": "missing",
                "failure_code": "RESOLUTION_DOWNLOAD_INFO_MISSING",
                "reason": "pip install record lacks download_info",
            }
        )

    raw_name = metadata.get("name")
    name = str(raw_name) if isinstance(raw_name, str) and raw_name else f"unknown-record-{index}"
    normalized_name = normalize_distribution_name(name)
    raw_version = metadata.get("version")
    version = str(raw_version) if isinstance(raw_version, str) and raw_version else "missing"

    url_parts = sanitized_url_parts(download_info.get("url"))
    hostname = str(url_parts["hostname"])
    expected = expected_authority(name)
    observed, authority_state = observed_authority(hostname)
    archive_info = download_info.get("archive_info")
    digest = archive_sha256(archive_info) if isinstance(archive_info, dict) else None

    record = {
        "record_index": index,
        "distribution_name": name[:120],
        "normalized_name": normalized_name,
        "version": version[:120],
        "expected_authority": expected,
        "observed_authority": observed,
        "authority_state": authority_state,
        "scheme": url_parts["scheme"],
        "hostname": hostname,
        "artifact_filename": url_parts["artifact_filename"],
        "path_prefix": url_parts["path_prefix"],
        "sanitized_url": url_parts["sanitized_url"],
        "url_sha256": url_parts["url_sha256"],
        "sha256": digest,
    }

    if url_parts["scheme"] != "https":
        violations.append(
            {
                "record_index": index,
                "distribution_name": normalized_name,
                "hostname": hostname,
                "failure_code": "ARTIFACT_URL_SCHEME_UNSAFE",
                "reason": "resolved artifact URL must use HTTPS",
            }
        )
    if url_parts["credentials_present"] is True:
        violations.append(
            {
                "record_index": index,
                "distribution_name": normalized_name,
                "hostname": hostname,
                "failure_code": "ARTIFACT_URL_CREDENTIALS_PRESENT",
                "reason": "resolved artifact URL embeds credentials",
            }
        )
    if url_parts["fragment_present"] is True:
        violations.append(
            {
                "record_index": index,
                "distribution_name": normalized_name,
                "hostname": hostname,
                "failure_code": "ARTIFACT_URL_FRAGMENT_PRESENT",
                "reason": "resolved artifact URL contains a fragment",
            }
        )
    if digest is None:
        violations.append(
            {
                "record_index": index,
                "distribution_name": normalized_name,
                "hostname": hostname,
                "failure_code": "ARTIFACT_SHA256_MISSING",
                "reason": "resolved artifact lacks one SHA-256 identity",
            }
        )
    if authority_state != "approved":
        violations.append(
            {
                "record_index": index,
                "distribution_name": normalized_name,
                "hostname": hostname,
                "failure_code": "ARTIFACT_HOST_REVIEW_REQUIRED",
                "reason": "resolved artifact host is not in the approved exact-host policy",
            }
        )
    elif observed != expected:
        violations.append(
            {
                "record_index": index,
                "distribution_name": normalized_name,
                "hostname": hostname,
                "failure_code": "PACKAGE_AUTHORITY_MISMATCH",
                "reason": (
                    f"expected authority {expected}, observed approved authority {observed}"
                ),
            }
        )

    return record, violations


def release_asset() -> tuple[str, str, str]:
    request = urllib.request.Request(
        VLLM_RELEASE_API,
        headers={
            "Accept": "application/vnd.github+json",
            "User-Agent": NOTEBOOK_NAME,
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    with urllib.request.urlopen(request, timeout=60.0) as response:
        payload = json.loads(response.read().decode("utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError("vLLM release metadata is not one JSON object")
    if payload.get("tag_name") != f"v{VLLM_RELEASE}":
        raise RuntimeError("vLLM release tag drifted")
    if payload.get("draft") is not False or payload.get("prerelease") is not False:
        raise RuntimeError("vLLM release must be published and non-prerelease")
    assets = payload.get("assets")
    if not isinstance(assets, list):
        raise RuntimeError("vLLM release assets are missing")
    candidates = [
        item
        for item in assets
        if isinstance(item, dict) and item.get("name") == VLLM_ASSET_NAME
    ]
    if len(candidates) != 1:
        raise RuntimeError("expected the exact official vLLM 0.19.1 x86_64 wheel")
    candidate = candidates[0]
    raw_url = candidate.get("browser_download_url")
    if not isinstance(raw_url, str):
        raise RuntimeError("vLLM release asset URL is missing")
    parsed = urllib.parse.urlsplit(raw_url)
    if (
        parsed.scheme != "https"
        or parsed.hostname != "github.com"
        or parsed.username is not None
        or parsed.password is not None
        or parsed.fragment
    ):
        raise RuntimeError("vLLM release asset URL violates the exact source contract")
    raw_digest = candidate.get("digest")
    if not isinstance(raw_digest, str) or not raw_digest.startswith("sha256:"):
        raise RuntimeError("vLLM release asset lacks a SHA-256 identity")
    digest = raw_digest.removeprefix("sha256:")
    if digest != VLLM_ASSET_SHA256:
        raise RuntimeError("vLLM release asset SHA-256 drifted")
    return str(candidate["name"]), raw_url, digest


def run_resolution(requirements_path: Path, report_path: Path) -> subprocess.CompletedProcess[str]:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--dry-run",
        "--ignore-installed",
        "--only-binary=:all:",
        "--report",
        str(report_path),
        "--index-url",
        PYPI_INDEX,
        "--extra-index-url",
        PYTORCH_INDEX,
        "-r",
        str(requirements_path),
    ]
    return subprocess.run(
        command,
        check=False,
        capture_output=True,
        text=True,
        timeout=1800.0,
        env={**os.environ, "PIP_DISABLE_PIP_VERSION_CHECK": "1"},
    )


def write_json(path: Path, payload: object) -> None:
    path.write_text(canonical_json(payload) + "\n", encoding="utf-8")


def main() -> int:
    present_credentials = [
        name for name in CREDENTIAL_ENV_NAMES if os.environ.get(name)
    ]
    if present_credentials:
        raise RuntimeError("credential-bearing environment variables are prohibited")

    if OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)
    OUTPUT_ROOT.mkdir(parents=True)

    historical_context = {
        "schema_version": "1.0.0",
        "historical_failures": HISTORICAL_FAILURES,
        "historical_runtime_context": HISTORICAL_RUNTIME_CONTEXT,
        "static_materializer_findings": STATIC_MATERIALIZER_FINDINGS,
    }
    write_json(OUTPUT_ROOT / "historical_context.json", historical_context)

    asset_name, asset_url, asset_digest = release_asset()
    requirements = (f"vllm @ {asset_url}", *FIXED_REQUIREMENTS)
    requirements_path = OUTPUT_ROOT / "requirements.in"
    requirements_path.write_text("\n".join(requirements) + "\n", encoding="utf-8")

    with tempfile.TemporaryDirectory(prefix="ag-vllm-recon-") as raw_temp:
        temp_root = Path(raw_temp)
        raw_report_path = temp_root / "pip-resolution-report.json"
        result = run_resolution(requirements_path, raw_report_path)
        stdout_excerpt = result.stdout[-12000:].replace(
            "/kaggle/working", "<working>"
        )
        stderr_excerpt = result.stderr[-12000:].replace(
            "/kaggle/working", "<working>"
        )
        command_evidence = {
            "returncode": result.returncode,
            "stdout_excerpt": stdout_excerpt,
            "stderr_excerpt": stderr_excerpt,
            "pip_download_command_performed": False,
            "package_installation_performed": False,
        }
        write_json(OUTPUT_ROOT / "resolution_command_evidence.json", command_evidence)

        if result.returncode != 0 or not raw_report_path.is_file():
            receipt = {
                "schema_version": "1.0.0",
                "notebook_name": NOTEBOOK_NAME,
                "status": "RESOLUTION_RECONNAISSANCE_BLOCKED",
                "failure_code": "PIP_RESOLUTION_FAILED",
                "resolved_distribution_count": 0,
                "host_count": 0,
                "policy_violation_count": 1,
                "wheel_files_written": 0,
                "package_installation_performed": False,
                "model_requests_performed": 0,
                "qualification_claimed": False,
            }
            write_json(OUTPUT_ROOT / "reconnaissance_receipt.json", receipt)
            print(canonical_json(receipt))
            return 0

        report = json.loads(raw_report_path.read_text(encoding="utf-8"))

    install_records = report.get("install")
    if not isinstance(install_records, list):
        install_records = []

    resolved_records: list[dict[str, object]] = []
    violations: list[dict[str, object]] = []
    for index, item in enumerate(install_records):
        record, record_violations = evaluate_install_record(item, index=index)
        resolved_records.append(record)
        violations.extend(record_violations)

    names = [str(record["normalized_name"]) for record in resolved_records]
    duplicate_names = sorted(
        name for name, count in Counter(names).items() if count > 1
    )
    for name in duplicate_names:
        violations.append(
            {
                "record_index": -1,
                "distribution_name": name,
                "hostname": "multiple",
                "failure_code": "DUPLICATE_DISTRIBUTION_IDENTITY",
                "reason": "resolved closure contains duplicate normalized distribution names",
            }
        )

    digests = [
        str(record["sha256"])
        for record in resolved_records
        if isinstance(record.get("sha256"), str)
    ]
    duplicate_digests = sorted(
        digest for digest, count in Counter(digests).items() if count > 1
    )
    for digest in duplicate_digests:
        violations.append(
            {
                "record_index": -1,
                "distribution_name": "multiple",
                "hostname": "multiple",
                "failure_code": "DUPLICATE_ARTIFACT_SHA256",
                "reason": f"duplicate artifact SHA-256: {digest}",
            }
        )

    resolved_records.sort(
        key=lambda record: (
            str(record["normalized_name"]),
            str(record["version"]),
        )
    )
    violations.sort(
        key=lambda violation: (
            str(violation["failure_code"]),
            str(violation["distribution_name"]),
            str(violation["hostname"]),
            int(violation["record_index"]),
        )
    )

    host_distributions: dict[str, list[str]] = defaultdict(list)
    authority_distributions: dict[str, list[str]] = defaultdict(list)
    for record in resolved_records:
        hostname = str(record["hostname"])
        normalized_name = str(record["normalized_name"])
        observed = str(record["observed_authority"])
        host_distributions[hostname].append(normalized_name)
        authority_distributions[observed].append(normalized_name)

    host_inventory = [
        {
            "hostname": hostname,
            "distribution_count": len(set(distributions)),
            "distributions": sorted(set(distributions)),
            "policy_state": (
                "approved"
                if hostname in APPROVED_HOST_AUTHORITIES
                else "review_required"
            ),
            "authority_candidate": (
                APPROVED_HOST_AUTHORITIES.get(hostname)
                or CANDIDATE_HOST_AUTHORITIES.get(hostname)
                or "unknown"
            ),
        }
        for hostname, distributions in sorted(host_distributions.items())
    ]
    authority_inventory = [
        {
            "authority": authority,
            "distribution_count": len(set(distributions)),
            "distributions": sorted(set(distributions)),
        }
        for authority, distributions in sorted(authority_distributions.items())
    ]

    sanitized_report = {
        "schema_version": "1.0.0",
        "pip_version": report.get("pip_version"),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "machine": platform.machine(),
        "vllm_asset_name": asset_name,
        "vllm_asset_sha256": asset_digest,
        "top_level_requirements": requirements,
        "resolved_artifacts": resolved_records,
    }
    write_json(
        OUTPUT_ROOT / "resolution_report.sanitized.json",
        sanitized_report,
    )
    write_json(
        OUTPUT_ROOT / "resolved_artifacts.json",
        {
            "schema_version": "1.0.0",
            "records": resolved_records,
        },
    )
    write_json(
        OUTPUT_ROOT / "host_inventory.json",
        {
            "schema_version": "1.0.0",
            "hosts": host_inventory,
        },
    )
    write_json(
        OUTPUT_ROOT / "authority_inventory.json",
        {
            "schema_version": "1.0.0",
            "authorities": authority_inventory,
        },
    )
    write_json(
        OUTPUT_ROOT / "policy_evaluation.json",
        {
            "schema_version": "1.0.0",
            "approved_exact_hosts": sorted(APPROVED_HOST_AUTHORITIES),
            "candidate_exact_hosts": sorted(CANDIDATE_HOST_AUTHORITIES),
            "violations": violations,
        },
    )

    status = (
        "RESOLUTION_RECONNAISSANCE_POLICY_COMPLETE"
        if not violations
        else "RESOLUTION_RECONNAISSANCE_REVIEW_REQUIRED"
    )
    receipt = {
        "schema_version": "1.0.0",
        "notebook_name": NOTEBOOK_NAME,
        "output_directory": OUTPUT_DIRECTORY_NAME,
        "status": status,
        "resolved_distribution_count": len(resolved_records),
        "host_count": len(host_inventory),
        "authority_count": len(authority_inventory),
        "policy_violation_count": len(violations),
        "historical_failure_count": len(HISTORICAL_FAILURES),
        "static_materializer_finding_count": len(STATIC_MATERIALIZER_FINDINGS),
        "wheel_files_written": len(tuple(OUTPUT_ROOT.rglob("*.whl"))),
        "pip_download_command_performed": False,
        "package_installation_performed": False,
        "model_loaded": False,
        "model_requests_performed": 0,
        "qualification_claimed": False,
        "authorization_issued": False,
        "customer_data_used": False,
        "credentials_used": False,
    }
    write_json(OUTPUT_ROOT / "reconnaissance_receipt.json", receipt)

    output_manifest_entries = []
    for path in sorted(OUTPUT_ROOT.iterdir(), key=lambda item: item.name):
        if not path.is_file():
            continue
        output_manifest_entries.append(
            {
                "path": path.name,
                "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                "size_bytes": path.stat().st_size,
            }
        )
    write_json(
        OUTPUT_ROOT / "output_sha256_manifest.json",
        {
            "schema_version": "1.0.0",
            "entries": output_manifest_entries,
        },
    )

    print(f"output_directory={OUTPUT_ROOT}")
    print(f"status={status}")
    print(f"resolved_distribution_count={len(resolved_records)}")
    print(f"host_count={len(host_inventory)}")
    print(f"policy_violation_count={len(violations)}")
    print("wheel_files_written=0")
    print("package_installation_performed=false")
    print("model_requests_performed=0")
    print("qualification_claimed=false")
    print("save_this_notebook_output=true")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
